In [ ]:
# @title 1.1 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev libtbb-dev libtbbmalloc2

# libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
!apt-get install -y -qq libcgal-dev libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.2 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml shapely mip POT


In [ ]:
%%bash
# @title 1.3 Installation de HGP-clusterer
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi



In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
# @title 1.4 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

print("Configuration CGAL...")

# -- FIX: Add CGAL path to environment for setup_cgal.py --
cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
# ---------------------------------------------------------

# On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
# car le setup.py principal pourrait réussir autrement.
try:
    subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
except subprocess.CalledProcessError:
    print("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
# Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
# Note: On le passe aussi explicitement ici pour être sûr
install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")


# Italian Olive Oil benchmark

Clustering with HGP-Clusterer on the Italian Olive Oil dataset.

In [ ]:
# @title 2.1 Load, Normalize and Cluster
import numpy as np
import pandas as pd
from hgp_clusterer import HGPClusterer
from scipy.optimize import linear_sum_assignment

# Load data from local CSV (already in the repository)
df = pd.read_csv('tests/OliveOil/oliveoil.csv')
features = ['palmitic', 'palmitoleic', 'stearic', 'oleic', 'linoleic', 'linolenic', 'arachidic', 'eicosenoic']
X = df[features].values.astype(float)

# L2 normalize over features (per column)
X = X / np.linalg.norm(X, axis=0, keepdims=True)

# Ground truth regions
y_str = df['region'].values
regions = ['Apulia.north', 'Calabria', 'Apulia.south', 'Sicily', 'Sardinia.inland', 'Sardinia.coast', 'Liguria.east', 'Liguria.west', 'Umbria']
y = np.array([regions.index(r) for r in y_str])

print(f"Data shape: {X.shape}")

# Run HGP-Clusterer
# We use the default backend (CGAL is selected)
# Note: For testing purposes if 8D Delaunay is too slow, complex_chosen='rips' could be added.
import sys
import time
print('------------------------------------------------------')
print(' [VERBOSE] Initialisation de HGPClusterer...')
print(f'   -> Backend sélectionné : CGAL')
print(f'   -> K = 2')
print(f'   -> Dimension des données = {X.shape[1]} (Attention : Dimension > 3)')
print('------------------------------------------------------')
sys.stdout.flush()
clusterer = HGPClusterer(backend='cgal', K=2, verbose=True, min_cluster_size=15, expZ=8)
print("Running fit_predict...")
print('\n[VERBOSE] Lancement de fit_predict()...')
print('⚠️ ATTENTION : Le calcul de la triangulation de Delaunay exacte')
print('en dimension 8 (avec CGAL) a une complexité exponentielle.')
print('Le processus C++ va générer des milliards de simplexes.')
print('Les logs C++ (verbose=True) peuvent ne pas s\'afficher en temps réel sur Colab.')
print('Veuillez patienter... (Cela peut prendre plusieurs heures)\n')
sys.stdout.flush()
start_time = time.time()
labels = clusterer.fit_predict(X)
print(f'\n[VERBOSE] fit_predict() terminé en {time.time() - start_time:.2f} secondes !')
sys.stdout.flush()

print(f"Found clusters: {np.unique(labels)}")

# Confusion matrix
n_clusters = labels.max() + 1
conf = np.zeros((9, n_clusters), dtype=int)
for i in range(len(y)):
    if labels[i] != -1:
        conf[y[i], labels[i]] += 1

# Hungarian Matching to ground truth (maximize intersection)
row_ind, col_ind = linear_sum_assignment(-conf)

# Our matched matrix (9x9)
ours_matrix = np.zeros((9, 9), dtype=int)
for r, c in zip(row_ind, col_ind):
    ours_matrix[r, c] = conf[r, c]

# Unclustered counts
ours_uncl = np.zeros(9, dtype=int)
for i in range(len(y)):
    if labels[i] == -1:
        ours_uncl[y[i]] += 1

# Persistable baseline
pers_matrix = [
    [21, 2, 0, 0, 0, 0, 0, 0, 0],
    [0, 50, 3, 0, 0, 0, 1, 0, 0],
    [0, 1, 196, 0, 0, 0, 0, 0, 0],
    [6, 20, 7, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 65, 0, 0, 0, 0],
    [0, 0, 0, 0, 2, 31, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 41, 2, 0],
    [0, 0, 0, 0, 0, 0, 0, 47, 0],
    [0, 0, 0, 0, 0, 0, 2, 0, 48]
]
pers_uncl = [2, 2, 9, 3, 0, 0, 7, 3, 1]

region_names = [
    "\\color[HTML]{CB0000}\\textbf{N.\\,Apulia}",
    "\\color[HTML]{FE0000}\\textbf{Calabria}",
    "\\color[HTML]{680100}\\textbf{S.\\,Apulia}",
    "\\color[HTML]{FD6864}\\textbf{Sicilia}",
    "\\color[HTML]{7B7B7B}\\textbf{Inl.\\,Sard.}",
    "\\color[HTML]{2F2F2F}\\textbf{Coast S.}",
    "\\color[HTML]{32CB00}\\textbf{E.\\,Ligur.}",
    "\\color[HTML]{34FF34}\\textbf{W.\\,Ligur.}",
    "\\color[HTML]{036400}\\textbf{Umbria}"
]

print("\nLaTeX Table:")
print("\\begin{table}[ht]")
print("\\centering")
print("\\caption{Results of the exhaustive versions of \\textsc{Persistable}")
print("and of our algorithm on the olive‐oil dataset.}")
print("\\label{tab:italie_deux}")
print("\\begin{tabular}{l*{9}{c}c}")
print("\\toprule")
print("\\textbf{\\nom{Pers.}/Ours} & \\textbf{1} & \\textbf{2} & \\textbf{3} & \\textbf{4} &")
print("\\textbf{5} & \\textbf{6} & \\textbf{7} & \\textbf{8} & \\textbf{9} & \\textbf{{\\footnotesize Unclustered}}\\\\")
print("\\midrule")

for i in range(9):
    if i in [4, 6]:
        print("\\midrule")
    elif i > 0:
        print("\\cmidrule(lr){2-11}")
    
    row_str = f"{region_names[i]}\n  "
    for j in range(9):
        p = pers_matrix[i][j]
        o = ours_matrix[i][j]
        if p == 0 and o == 0:
            if j == 3: # In original table, column 4 is often just '--'
                row_str += "& -- %\n  "
            else:
                row_str += "& \\color[HTML]{C0C0C0}0/0 "
        else:
            p_str = f"\\textbf{{{p}}}" if p > o else str(p)
            o_str = f"\\textbf{{{o}}}" if o > p else str(o)
            if p == o:
                p_str = f"\\textbf{{{p}}}"
                o_str = f"\\textbf{{{o}}}"
            
            if j == 3 and p == 0 and o == 0:
                row_str += "& -- %\n  "
            else:
                row_str += f"& {p_str}/{o_str} "
    
    p_u = pers_uncl[i]
    o_u = ours_uncl[i]
    if p_u == 0 and o_u == 0:
        row_str += "& \\color[HTML]{C0C0C0}0/0\\\\"
    else:
        pu_str = f"\\textbf{{{p_u}}}" if p_u < o_u else str(p_u)
        ou_str = f"\\textbf{{{o_u}}}" if o_u < p_u else str(o_u)
        if p_u == o_u:
            pu_str = str(p_u)
            ou_str = str(o_u)
        row_str += f"& {pu_str}/{ou_str}\\\\"
    
    print(row_str)

print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")
